In [0]:
#Khai báo thư viện và Cấu hình đường dẫn tập trung

import warnings
warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from mlflow.models import infer_signature
import mlflow

spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

# ==============================================================================
# ⚙️ CẤU HÌNH ĐƯỜNG DẪN PATHS (Thay đổi các giá trị này theo cấu trúc máy của bạn)
# ==============================================================================
# 1. Đường dẫn tới file dữ liệu tầng Silver đầu vào
INPUT_SILVER_CSV_PATH = "/Volumes/workspace/smart_gpa/dataset/diem_sinh_vien_silver.csv"

# 2. Thư mục tạm thời của Volume dùng cho lưu trữ MLflow (Tránh lỗi JVM Serverless)
MLFLOW_TMP_DIR       = "/Volumes/workspace/smart_gpa/dataset/mlflow_tmp_silver_fail"

# 3. Tên định danh đầy đủ của mô hình khi đăng ký lên Unity Catalog
# Định dạng: <tên_catalog>.<tên_schema_database>.<tên_mô_hình>
UNITY_CATALOG_MODEL_NAME = "workspace.smartgpa_db.silver_failure_model"
# ==============================================================================

In [0]:
#Nạp dữ liệu và Tiền xử lý (Sử dụng biến Path)

# 1. Đọc dữ liệu từ đường dẫn cấu hình trên cùng
df_silver_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(INPUT_SILVER_CSV_PATH)

# 2. Mã hóa cột chữ 'loai_hoc_phan'
indexer = StringIndexer(inputCol="loai_hoc_phan", outputCol="loai_hoc_phan_encoded")
df_indexed = indexer.fit(df_silver_raw).transform(df_silver_raw)

# 3. Định nghĩa nhãn mục tiêu (Label)
df_labeled = df_indexed.withColumn(
    "label",
    when(col("status_canh_bao") == "Nguy co rot mon", 1.0).otherwise(0.0)
)

# 4. Gom tất cả các cột tính năng vào Vector
feature_cols = [
    "so_tiet_lt", "so_tiet_th", "so_chi_lt", "so_chi_th", "tong_so_chi", 
    "diem_trung_binh_thuc_hanh", "diem_tich_luy_hien_tai", "loai_hoc_phan_encoded"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

df_final_ml = assembler.transform(df_labeled).select("features", "label")
df_final_ml.show(5, truncate=False)

+-----------------------------------+-----+
|features                           |label|
+-----------------------------------+-----+
|[45.0,0.0,3.0,0.0,3.0,2.9,8.0,0.0] |0.0  |
|[30.0,30.0,2.0,1.0,3.0,8.1,6.3,1.0]|0.0  |
|[45.0,0.0,3.0,0.0,3.0,3.9,4.4,0.0] |0.0  |
|[30.0,30.0,2.0,1.0,3.0,3.0,9.1,1.0]|0.0  |
|[45.0,0.0,3.0,0.0,3.0,9.0,8.5,0.0] |0.0  |
+-----------------------------------+-----+
only showing top 5 rows


In [0]:
#Huấn luyện, Ghi nhận MLflow và Đăng ký mô hình lên Unity Catalog
# Chia tập dữ liệu Train/Test
train_data, test_data = df_final_ml.randomSplit([0.8, 0.2], seed=42)

# Khởi tạo mô hình Random Forest
rf_model = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=6, seed=42)

with mlflow.start_run(run_name="Silver_Subject_Failure_Prediction") as run:
    # Huấn luyện
    trained_rf = rf_model.fit(train_data)
    
    # Dự đoán thử nghiệm
    predictions = trained_rf.transform(test_data)
    
    # Đánh giá độ chính xác (Accuracy)
    evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
    accuracy = evaluator.evaluate(predictions)
    print(f"🎯 Độ chính xác (Accuracy) của mô hình: {round(accuracy * 100, 2)}%")
    
    # Tạo Model Signature chuẩn hóa API đầu vào/đầu ra
    X_train = train_data.select("features")
    y_pred = predictions.select("prediction")
    signature = infer_signature(X_train, y_pred)
    
    # Lưu log mô hình sử dụng biến thư mục tạm cấu hình ở Cell 1
    mlflow.spark.log_model(
        spark_model=trained_rf,
        artifact_path="model",
        signature=signature,
        dfs_tmpdir=MLFLOW_TMP_DIR
    )
    
    run_id = run.info.run_id

# Đăng ký mô hình lên Unity Catalog sử dụng tên cấu hình ở Cell 1
registered_model = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model", 
    name=UNITY_CATALOG_MODEL_NAME
)
print(f"✅ Mô hình đã được ghi nhận thành công tại Catalog: {registered_model.name} (Version {registered_model.version})")

🎯 Độ chính xác (Accuracy) của mô hình: 98.37%


2026/05/31 15:58:10 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.6) contains a local version label (+databricks.connect.18.0.6). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/31 15:58:13 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-f50ce551-32c2-4c98-adb9-aa/tmpwmee8szz/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
Successfully registered model 'workspace.smartgpa_db.silver_failure_model'.


Uploading artifacts:   0%|          | 0/24 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.smartgpa_db.silver_failure_model': https://dbc-f659bab4-1de9.cloud.databricks.com/explore/data/models/workspace/smartgpa_db/silver_failure_model/version/1?o=7474656584148137


✅ Mô hình đã được ghi nhận thành công tại Catalog: workspace.smartgpa_db.silver_failure_model (Version 1)
